# Notebook 03 — Supervised Machine Learning: Career Prediction & Model Comparison

**Project:** A Data-Driven Job Market Skill Demand Analysis and Career Recommendation System Using NLP and Machine Learning

This notebook adds a supervised machine-learning component to the existing project. It uses the same `job_postings.csv` dataset and treats **career/title as the prediction target**. Several classification models are trained and compared, followed by a user-input career prediction example.

**Models compared:** Logistic Regression, Random Forest, Support Vector Machine (SVM), and Gradient Boosting.

## 1. Import Libraries

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

## 2. Load Dataset

In [ ]:
DATA_PATH = 'job_postings.csv'
if not os.path.exists(DATA_PATH): DATA_PATH = '../data/job_postings.csv'
if not os.path.exists(DATA_PATH): raise FileNotFoundError('job_postings.csv was not found. Place it beside this notebook or update DATA_PATH.')
df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
display(df.head())

## 3. Inspect Target and Candidate Features

In [ ]:
print('Columns:', df.columns.tolist())
print('\nCareer distribution:')
display(df['title'].value_counts().to_frame('count'))
print('\nMissing values:')
display(df.isna().sum().to_frame('missing_count'))

## 4. Prepare Prediction Features

**Target:** `title` (career category).

**User-enterable features:** required skills, experience level, education, industry, remote type, and salary range.

Job ID, company, location, company size, description, posted date, applications, and days to fill are excluded from the supervised prediction model because these are job-market attributes rather than information a typical user would provide.


In [ ]:
ml_df = df[['title','required_skills','experience_level','education_required','industry','remote_type','salary_min','salary_max']].copy()

categorical_cols = ['experience_level','education_required','industry','remote_type']
numeric_cols = ['salary_min','salary_max']

for col in categorical_cols:
    ml_df[col] = ml_df[col].fillna('unknown').astype(str).str.strip().str.lower()

for col in numeric_cols:
    ml_df[col] = pd.to_numeric(ml_df[col], errors='coerce')
    ml_df[col] = ml_df[col].fillna(ml_df[col].median())

ml_df['skill_list'] = ml_df['required_skills'].fillna('').astype(str).apply(
    lambda x: [s.strip().lower() for s in x.split('|') if s.strip()]
)

print('Prepared dataset:', ml_df.shape)
display(ml_df.head())


## 5. Convert Skills and Categorical Features into Model Features

In [ ]:
mlb = MultiLabelBinarizer()
skill_matrix = mlb.fit_transform(ml_df['skill_list'])

skill_columns = [f'skill__{s}' for s in mlb.classes_]
skill_df = pd.DataFrame(
    skill_matrix,
    columns=skill_columns,
    index=ml_df.index
)

category_df = pd.get_dummies(
    ml_df[categorical_cols],
    prefix=categorical_cols,
    dtype=int
)

numeric_df = ml_df[numeric_cols].copy()

X = pd.concat(
    [skill_df, category_df, numeric_df],
    axis=1
)

y = ml_df['title']

print('Skill features:', len(skill_columns))
print('Total features:', X.shape[1])
print('Career classes:', sorted(y.unique()))


## 6. Stratified Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## 7. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Training feature matrix:', X_train_scaled.shape)

## 8. Train and Compare Classification Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    'SVM': SVC(probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=3, random_state=42)
}
results=[]
trained_models={}
for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_scaled, y_train)
    pred=model.predict(X_test_scaled)
    results.append({'Model':name,'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred,average='macro',zero_division=0),'Recall':recall_score(y_test,pred,average='macro',zero_division=0),'F1-Score':f1_score(y_test,pred,average='macro',zero_division=0)})
    trained_models[name]=model
results_df=pd.DataFrame(results).sort_values('F1-Score',ascending=False).reset_index(drop=True)
display(results_df)

## 9. Model Comparison Visualization

In [ ]:
comparison_plot=results_df.set_index('Model')[['Accuracy','Precision','Recall','F1-Score']]
ax=comparison_plot.plot(kind='bar',figsize=(11,6))
ax.set_title('Machine Learning Model Comparison')
ax.set_ylabel('Score')
ax.set_xlabel('Model')
ax.set_ylim(0,1.05)
plt.xticks(rotation=20,ha='right')
plt.tight_layout()
plt.show()

## 10. Select the Best Model

In [ ]:
best_model_name=results_df.iloc[0]['Model']
best_model=trained_models[best_model_name]
print('Best model based on macro F1-score:',best_model_name)
display(results_df)

## 11. Detailed Evaluation of the Best Model

In [ ]:
best_pred=best_model.predict(X_test_scaled)
print(classification_report(y_test,best_pred,digits=4,zero_division=0))

## 12. Confusion Matrix

In [ ]:
cm=confusion_matrix(y_test,best_pred,labels=best_model.classes_)
fig,ax=plt.subplots(figsize=(12,10))
disp=ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=best_model.classes_)
disp.plot(ax=ax,xticks_rotation=45,values_format='d')
ax.set_title(f'Confusion Matrix — {best_model_name}')
plt.tight_layout()
plt.show()

## 13. User Input Career Prediction

Edit the example values to test a different user profile.

In [ ]:
def build_user_features(
    skills,
    experience_level,
    education_required,
    industry,
    remote_type,
    salary_min,
    salary_max
):
    row = pd.DataFrame(
        0,
        index=[0],
        columns=X.columns
    )

    for skill in skills:
        col = f"skill__{str(skill).strip().lower()}"
        if col in row.columns:
            row.loc[0, col] = 1

    cat_values = {
        'experience_level': experience_level,
        'education_required': education_required,
        'industry': industry,
        'remote_type': remote_type
    }

    for col, value in cat_values.items():
        feature = f"{col}_{str(value).strip().lower()}"
        if feature in row.columns:
            row.loc[0, feature] = 1

    row.loc[0, 'salary_min'] = salary_min
    row.loc[0, 'salary_max'] = salary_max

    return row

example_user = build_user_features(
    ['Python','SQL','Machine Learning','Statistics','TensorFlow'],
    'mid',
    'master',
    'tech',
    'hybrid',
    120000,
    160000
)

example_scaled = scaler.transform(example_user)
predicted_career = best_model.predict(example_scaled)[0]
print('Predicted Career:', predicted_career)

if hasattr(best_model, 'predict_proba'):
    probabilities = best_model.predict_proba(example_scaled)[0]
    top = np.argsort(probabilities)[::-1][:3]
    top_predictions = pd.DataFrame({
        'Career': best_model.classes_[top],
        'Probability (%)': np.round(probabilities[top] * 100, 2)
    })
    display(top_predictions)


## 14. Test Another User Profile

In [ ]:
user_profile = build_user_features(
    ['Python','SQL','AWS','Docker'],
    'senior',
    'bachelor',
    'tech',
    'remote',
    130000,
    175000
)

user_scaled = scaler.transform(user_profile)
predicted = best_model.predict(user_scaled)[0]
print('Predicted Career:', predicted)

if hasattr(best_model, 'predict_proba'):
    probs = best_model.predict_proba(user_scaled)[0]
    top3 = np.argsort(probs)[::-1][:3]
    display(pd.DataFrame({
        'Career': best_model.classes_[top3],
        'Probability (%)': np.round(probs[top3] * 100, 2)
    }))


## 15. Save the Best Model for Flask Integration

In [ ]:
MODEL_ARTIFACT = {
    'model': best_model,
    'scaler': scaler,
    'columns': X.columns.tolist(),
    'skill_columns': skill_columns,
    'categorical_cols': categorical_cols,
    'numeric_cols': numeric_cols,
    'best_model_name': best_model_name,
    'target_classes': best_model.classes_.tolist()
}

# Save the complete model artifact where the Flask backend expects it.
MODEL_DIR = os.path.join('models')
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODEL_DIR, 'career_prediction_svm.pkl')

joblib.dump(MODEL_ARTIFACT, MODEL_PATH)

print('Saved:', os.path.abspath(MODEL_PATH))

## 16. Summary

This notebook adds a supervised machine-learning component to the project. It compares multiple classification models, evaluates them using accuracy, macro precision, macro recall, and macro F1-score, and demonstrates career prediction using features that a typical user can provide: skills, experience level, education, industry, remote preference, and salary range. Job-market attributes such as applications and days to fill remain available for market analysis but are excluded from the personal career prediction model. The saved SVM model is integrated into the Flask backend.
